In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_deputados_ingest
# LOCAL: 1_bronze/src/
# OBJETIVO: Dados brutos deputados
# ----------------------------------------------------------------------------------------

import requests as req

# Acesso a API e recuperação dos dados
response = req.get("https://dadosabertos.camara.leg.br/api/v2/deputados").json()
dados_lista = response['dados'] 

# Dataframe para salvar os dados
df_deputados = spark.createDataFrame(dados_lista)

# Salvando os dados na tabela da camada bronze
df_deputados.write.mode("overwrite").saveAsTable("bronze_deputados")


display(spark.table("bronze_deputados"))


In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_deputados_ingest
# LOCAL: 1_bronze/src/
# OBJETIVO: Dados brutos deputados (versão com paginação)
# ----------------------------------------------------------------------------------------


import requests as req

pagina = 1
todos_deputados = []
itens_por_pagina = 100 # Definir um valor alto agiliza o processo

while True:
    # 1. Montar a URL com os parâmetros de paginação
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados?pagina={pagina}&itens={itens_por_pagina}&ordem=ASC&ordenarPor=nome"
    
    try:
        response = req.get(url).json()
        dados_da_pagina = response['dados'] 

        if not dados_da_pagina:
            break
                    
        todos_deputados.extend(dados_da_pagina)
        
        print(f"Página {pagina} capturada...")
        pagina += 1
        
    except Exception as e:
        print(f"Erro na página {pagina}: {e}")
        break

# Criar o DataFrame com a lista COMPLETA (todos_deputados)
if todos_deputados:
    df_deputados = spark.createDataFrame(todos_deputados)

    # Salvar na Bronze
    df_deputados.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_deputados")
    
    print(f"Total baixado: {len(todos_deputados)} deputados em {pagina-1} páginas.")
else:
    print("Nenhum dado capturado.")

display(spark.table("bronze_deputados"))